# 05c — Age clock: predicting age in months

Ridge regression on adults only (3m and older), leave-one-mouse-out. Judged per mouse (median of its cells),
against a naive baseline (always guess the average age of the other mice).
Needs `figure/{TISSUE}/age_auroc.csv` from 05a. Findings at the end are Lung's.

### 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from src import aging as ag    # shared functions: edit src/aging.py, not the notebooks

TISSUE = os.environ.get("TISSUE", "Lung")   # set by src/run_tissue.py; edit here to run by hand
P = ag.paths(TISSUE)                        # data/{TISSUE}/..., figure/{TISSUE}/...

adata = sc.read_h5ad(P.processed)

res = pd.read_csv(P.figure("age_auroc.csv"), index_col=0)
good = list(res.index[res["logreg"] >= 0.8])
clean = ag.feature_mask(adata, "clean")

### 2. Clock per cell type, three regularization strengths

Report all three; picking the best after seeing test results inflates the score.

In [ ]:
tables = []
for alpha in [10, 100, 1000]:
    t = pd.DataFrame({ct: ag.clock_summary(adata, ct, alpha=alpha, features=clean, **ag.main_kw(ct, TISSUE))
                      for ct in good}).T
    t["alpha"] = alpha
    tables.append(t)
if good:
    clock = pd.concat(tables).astype(float).round(2)
    clock["beats_naive"] = clock["mae_months"] < clock["naive_mae"]
else:
    print("No cell type reaches AUROC 0.8 in", TISSUE, "- no clock is fit.")
    clock = pd.DataFrame(columns=["mae_months", "naive_mae", "spearman_mice", "n_mice", "alpha", "beats_naive"])
clock.to_csv(P.figure("clock_summary.csv"))
clock.sort_values(["alpha", "mae_months"])

### 3. Predicted vs. true age, one cell type

The tissue's reviewed choice if it has one (Lung: alveolar macrophage), otherwise the lowest error at `alpha=100`.

In [ ]:
if good:
    a100 = clock[clock["alpha"] == 100]
    ct = ag.settings(TISSUE).get("example_clock") or a100["mae_months"].idxmin()
    cells, pm = ag.clock_eval(adata, ct, features=clean, **ag.main_kw(ct, TISSUE))
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.boxplot(data=cells, x="true", y="pred", ax=ax, color="lightgray", showfliers=False)
    sns.stripplot(data=pm, x="true", y="pred", ax=ax, color="black", size=8, jitter=0.1)
    ax.set_xlabel("true age (months)"); ax.set_ylabel("predicted age (months)")
    ax.set_title(f"{ct}: gray = cells, black = mouse median")
    fig.savefig(P.figure(f"clock_{ag.slug(ct)}.png"), dpi=200, bbox_inches="tight")

### 4. Held-out age groups

Each old age group that lies between other adult ages (Lung: 18m and 21m) is held out in turn. A real clock places a held-out mouse between its neighboring ages (Lung: 21m between 18 and 30, 18m between 3 and 21).

In [ ]:
adult = [a for a in ag.ages_in(adata) if a in ag.ADULT]
holds = [a for a in adult[1:-1] if a in ag.OLD]
rows = []
for ct in good:
    for hold in holds:
        pred = ag.held_out_age(adata, ct, hold=hold, features=clean, **ag.main_kw(ct, TISSUE))
        rows.append({"cell_type": ct, "held_out": hold, "median_pred": round(pred.median(), 1),
                     "per_mouse": pred.to_dict()})
held = pd.DataFrame(rows)
held.to_csv(P.figure("clock_held_out_age.csv"), index=False)
held

### Findings (Lung)

- **Error per mouse** is 4.4–8.0 months at `alpha=100`, against a naive baseline of 7.3–9.7. All nine cell types beat the baseline at `alpha=10` and `alpha=100`, and eight do at `alpha=1000`, but neutrophils only barely.
- **The result depends on `alpha`,** so all three are reported. Alveolar macrophage has the lowest error, but that cell type doesn't pass the batch check, so its ranking is not evidence of a better clock.
- **The clock doesn't resolve ages within the old range.** Held-out 21m mice are placed at a median of 9.8–15.8 months depending on cell type, younger than the held-out 18m mice in every cell type. Both 21m mice are female and from their own batch.